In [2]:
!pip install nltk transformers gradio

In [3]:
import re
import nltk
import gradio as gr
from transformers import pipeline
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [4]:
glossary = {
    # Signal Processing / Math
    "Fourier Transform": "a method to break signals into frequencies",
    "Convolution": "a way to combine two signals or functions",
    "Entropy": "a measure of disorder or randomness",
    "Eigenvalue": "a special number that shows how a transformation stretches a vector",

    # Core CS concepts
    "Algorithm": "a step-by-step method to solve a problem",
    "Data Structure": "a way to organize and store data",
    "Recursion": "a method where a function calls itself",
    "Hashing": "a technique to map data to fixed-size values for quick lookups",
    "Pointer": "a variable that stores the memory address of another variable",
    "Deadlock": "a state where two processes wait forever for each other",
    "Semaphore": "a mechanism to control access to shared resources in concurrency",
    "Virtual Memory": "a memory management technique that uses disk storage like RAM",

    # Networking
    "TCP/IP": "a set of rules for communication between computers over networks",
    "DNS": "a system that translates website names to IP addresses",
    "Packet Switching": "a method of sending data in small chunks across a network",
    "Firewall": "a system that blocks or allows network traffic for security",

    # Databases
    "Normalization": "a process to organize database tables to remove redundancy",
    "ACID": "a set of rules (Atomicity, Consistency, Isolation, Durability) for reliable transactions",
    "Indexing": "a method to make data retrieval faster in databases",

    # AI/ML
    "Neural Network": "a machine learning model inspired by the brain",
    "Backpropagation": "a method to adjust neural network weights by learning from errors",
    "Gradient Descent": "an algorithm to minimize error by adjusting parameters step by step",
    "Overfitting": "when a model memorizes training data but fails on new data",
    "Regularization": "a technique to prevent overfitting by adding constraints",
    "Transformer Model": "a neural network architecture that processes sequences in parallel (used in ChatGPT)",
    "Embedding": "a way to represent words or items as numeric vectors",

    # Software Engineering
    "Agile": "a flexible method of software development with short iterations",
    "Scrum": "a framework in Agile where tasks are managed in sprints",
    "DevOps": "a culture combining development and operations for faster delivery",
    "Version Control": "a system to track changes in code (like Git)",
    "CI/CD": "Continuous Integration/Continuous Deployment, automating build and deployment",

    # Security
    "Cryptography": "the science of securing communication using codes",
    "Public Key": "a cryptographic key shared openly to encrypt data",
    "Private Key": "a secret key used to decrypt data in cryptography",
    "Hash Function": "a function that converts input into a fixed-length code",

    "Algorithm": "a step-by-step method to solve a problem",
    "Data Structure": "a way to organize and store data",
    "Array": "a collection of items stored at contiguous memory locations",
    "Linked List": "a data structure made of nodes where each node points to the next one",
    "Stack": "a data structure that follows last in, first out (LIFO) order",
    "Queue": "a data structure that follows first in, first out (FIFO) order",
    "Hash Table": "a structure that maps keys to values for quick lookup",
    "Tree": "a hierarchical structure with a root and child nodes",
    "Binary Tree": "a tree where each node has at most two children",
    "Binary Search Tree": "a tree where left child values are smaller and right child values are larger",
    "Heap": "a tree-based structure where parent nodes are ordered with respect to their children",
    "Graph": "a set of nodes connected by edges",
    "Directed Graph": "a graph where edges have a direction",
    "Undirected Graph": "a graph where edges have no direction",
    "Adjacency Matrix": "a way to represent a graph using a 2D array",
    "Adjacency List": "a way to represent a graph using lists of neighbors",
    "Depth First Search": "a graph traversal method exploring as far as possible along each branch",
    "Breadth First Search": "a graph traversal method visiting all neighbors level by level",
    "Dynamic Programming": "a method of solving problems by breaking them into overlapping subproblems",
    "Greedy Algorithm": "an approach that makes the best choice at each step",
    "Divide and Conquer": "an approach that splits problems into subproblems, solves them, then combines results",
    "Sorting": "arranging elements in order",
    "Merge Sort": "a sorting algorithm using divide and conquer",
    "Quick Sort": "a fast sorting algorithm using partitioning",
    "Bubble Sort": "a simple sorting algorithm repeatedly swapping adjacent elements",
    "Selection Sort": "a simple sorting algorithm selecting the smallest element each time",
    "Insertion Sort": "a sorting algorithm inserting elements into their correct position",
    "Counting Sort": "a sorting algorithm using counts of elements",
    "Radix Sort": "a non-comparison sort using digit positions",
    "Bucket Sort": "a sorting method distributing elements into buckets then sorting each bucket",
    "Binary Search": "an algorithm to find an item in a sorted array by dividing in half",
    "Linear Search": "an algorithm to check each item until the target is found",
    "Big O Notation": "notation to describe the upper bound of algorithm complexity",
    "Big Theta Notation": "notation describing tight bound of algorithm complexity",
    "Big Omega Notation": "notation describing best-case algorithm complexity",
    "NP Problem": "a problem verifiable in polynomial time",
    "NP Complete": "problems as hard as any NP problem",
    "NP Hard": "problems at least as hard as NP complete problems",
    "Compiler": "software that translates code into machine language",
    "Interpreter": "software that executes code line by line",
    "Virtual Machine": "software that emulates a physical computer",
    "Assembler": "translates assembly language into machine code",
    "Linker": "combines object files into a single executable",
    "Loader": "loads programs into memory for execution",
    "Syntax": "rules defining how programs are written",
    "Semantic": "meaning of program instructions",
    "Lexical Analysis": "breaking source code into tokens",
    "Parsing": "analyzing tokens against grammar rules",
    "Abstract Syntax Tree": "a tree representation of program structure",
    "Intermediate Code": "code between high-level and machine code",
    "Optimization": "improving code for performance",
    "Machine Code": "binary instructions executed by a CPU",
    "Assembly Language": "low-level programming language close to machine code",
    "High-Level Language": "programming languages closer to human language",
    "Object-Oriented Programming": "a paradigm using objects and classes",
    "Class": "a blueprint for creating objects",
    "Object": "an instance of a class",
    "Method": "a function defined inside a class",
    "Inheritance": "a way for a class to use properties of another class",
    "Polymorphism": "ability to use one interface with many forms",
    "Encapsulation": "hiding internal details of a class",
    "Abstraction": "focusing on essential features, hiding details",
    "Interface": "a contract defining methods without implementation",
    "Constructor": "a special method to initialize objects",
    "Destructor": "a method called when an object is destroyed",
    "Overloading": "defining multiple methods with same name but different arguments",
    "Overriding": "redefining a method in a subclass",
    "Exception Handling": "managing errors gracefully in programs",
    "Try Catch": "a block to handle exceptions",
    "Multithreading": "executing multiple threads simultaneously",
    "Process": "an independent program in execution",
    "Thread": "a lightweight process",
    "Concurrency": "managing multiple tasks at the same time",
    "Parallelism": "executing tasks simultaneously on multiple processors",
    "Deadlock": "a state where processes wait indefinitely",
    "Starvation": "a process never getting needed resources",
    "Semaphore": "a synchronization tool to control access",
    "Mutex": "a lock allowing only one thread access at a time",
    "Critical Section": "a part of code that should not be executed by more than one thread at a time",
    "Operating System": "software that manages hardware and software resources",
    "Kernel": "the core of an operating system",
    "System Call": "a program request to the operating system",
    "Process Scheduling": "deciding which process gets CPU time",
    "Round Robin": "a scheduling algorithm giving equal time slices",
    "First Come First Serve": "a scheduling algorithm serving in arrival order",
    "Shortest Job First": "a scheduling algorithm choosing shortest task first",
    "Paging": "a memory management scheme dividing memory into pages",
    "Segmentation": "a memory management technique dividing programs into segments",
    "Virtual Memory": "using disk space as memory",
    "Cache Memory": "small fast memory between CPU and RAM",
    "Primary Memory": "main memory (RAM)",
    "Secondary Memory": "storage devices like hard drives",
    "File System": "method of organizing and storing files",
    "Database": "a structured collection of data",
    "Relational Database": "a database based on tables and relations",
    "SQL": "language for managing relational databases",
    "Primary Key": "a unique identifier for table records",
    "Foreign Key": "a key linking two tables",
    "Index": "a database tool for faster retrieval",
    "Normalization": "organizing data to reduce redundancy",
    "Denormalization": "combining tables to improve performance",
    "Transaction": "a unit of work in a database",
    "ACID": "properties ensuring reliable transactions",
    "BASE": "properties of NoSQL databases (Basically Available, Soft state, Eventual consistency)",
    "NoSQL": "databases not based on tables",
    "Document Database": "a database storing data as documents",
    "Key-Value Store": "a database storing pairs of keys and values",
    "Column Store": "a database storing data by columns",
    "Graph Database": "a database using nodes and edges",
    "CAP Theorem": "consistency, availability, partition tolerance tradeoff in databases",
    "Data Warehouse": "a large storage of integrated data",
    "Data Mining": "discovering patterns in large datasets",
    "Machine Learning": "teaching computers to learn from data",
    "Supervised Learning": "learning with labeled data",
    "Unsupervised Learning": "learning without labeled data",
    "Reinforcement Learning": "learning through rewards and punishments",
    "Regression": "predicting continuous values",
    "Classification": "predicting categories",
    "Clustering": "grouping similar data points",
    "Decision Tree": "a model using branches for decisions",
    "Random Forest": "a collection of decision trees",
    "Support Vector Machine": "a classifier using hyperplanes",
    "Naive Bayes": "a classifier using Bayes theorem",
    "K Nearest Neighbors": "a classifier using nearest data points",
    "Neural Network": "a model inspired by the human brain",
    "Activation Function": "a function controlling neuron output",
    "Backpropagation": "a method to train neural networks",
    "Gradient Descent": "an optimization method minimizing error",
    "Overfitting": "when a model memorizes training data",
    "Underfitting": "when a model is too simple to capture patterns",
    "Regularization": "a technique to prevent overfitting",
    "Dropout": "a method turning off random neurons in training",
    "Batch Normalization": "a method to normalize layer inputs",
    "Convolutional Neural Network": "a network good for image data",
    "Recurrent Neural Network": "a network good for sequence data",
    "LSTM": "a recurrent neural network handling long dependencies",
    "GRU": "a simpler recurrent neural network variant",
    "Transformer": "a model handling sequences in parallel",
    "Attention Mechanism": "a method to focus on important parts of data",
    "Embedding": "a numeric representation of words or data",
    "Word2Vec": "a model to learn word embeddings",
    "GloVe": "an embedding model based on co-occurrence",
    "BERT": "a transformer model for language understanding",
    "GPT": "a transformer model generating text",
    "Pretraining": "training a model on large data first",
    "Fine-Tuning": "adjusting a pretrained model to specific tasks",
    "Transfer Learning": "using knowledge from one task in another",
    "Reinforcement Signal": "reward or penalty in reinforcement learning",
    "Exploration": "trying new actions in reinforcement learning",
    "Exploitation": "using known best actions in reinforcement learning",
    "Markov Decision Process": "a model for reinforcement learning problems",
    "Policy": "strategy in reinforcement learning",
    "Value Function": "expected reward in reinforcement learning",
    "Q Learning": "a reinforcement learning algorithm",
    "Deep Q Network": "a deep learning method for Q learning",
    "Computer Network": "a group of computers connected to share data",
    "LAN": "local area network",
    "WAN": "wide area network",
    "MAN": "metropolitan area network",
    "Topology": "arrangement of network elements",
    "Bus Topology": "a single cable connecting devices",
    "Star Topology": "devices connected to a central hub",
    "Ring Topology": "devices connected in a circle",
    "Mesh Topology": "devices interconnected with many paths",
    "OSI Model": "a 7-layer model for networking",
    "TCP IP Model": "a 4-layer networking model",
    "Physical Layer": "layer handling hardware connections",
    "Data Link Layer": "layer handling error detection and framing",
    "Network Layer": "layer handling routing and addressing",
    "Transport Layer": "layer ensuring reliable delivery",
    "Session Layer": "layer managing sessions",
    "Presentation Layer": "layer handling data translation and encryption",
    "Application Layer": "layer where user applications work",
    "TCP": "a reliable transport protocol",
    "UDP": "a faster but unreliable transport protocol",
    "IP Address": "a unique identifier for devices on a network",
    "IPv4": "an internet protocol with 32-bit addresses",
    "IPv6": "an internet protocol with 128-bit addresses",
    "Subnetting": "dividing a network into smaller networks",
    "DNS": "system converting names to IP addresses",
    "DHCP": "protocol assigning IP addresses automatically",
    "Firewall": "a system controlling network access",
    "Proxy Server": "a server acting as intermediary",
    "VPN": "a secure private network over the internet",
    "Cloud Computing": "delivering services over the internet",
    "IaaS": "infrastructure as a service",
    "PaaS": "platform as a service",
    "SaaS": "software as a service",
    "Virtualization": "running multiple systems on one hardware",
    "Containerization": "running apps in isolated environments",
    "Docker": "a tool for containerization",
    "Kubernetes": "a system for managing containers",

    "Operating System": "software that manages hardware and software resources",
    "Real Time OS": "an OS that responds to events within strict time limits",
    "Embedded System": "a computer system with a dedicated function within a larger device",
    "Microcontroller": "a compact integrated circuit with CPU, memory, and peripherals",
    "Microprocessor": "a CPU on a single integrated circuit",
    "BIOS": "basic input output system used to initialize hardware",
    "Device Driver": "software that controls hardware devices",
    "Firmware": "permanent software programmed into hardware",
    "Cloud Storage": "storing data on remote servers accessed via the internet",
    "Edge Computing": "processing data closer to the data source",
    "Fog Computing": "a distributed computing model between cloud and edge",
    "Server": "a computer that provides services to clients",
    "Client": "a computer requesting services from a server",
    "Peer to Peer": "a network where computers act as both clients and servers",
    "Load Balancer": "a system that distributes traffic across servers",
    "Scalability": "the ability of a system to handle growth",
    "Fault Tolerance": "the ability to continue working despite failures",
    "High Availability": "systems designed to be operational most of the time",
    "Distributed System": "a system with components located on networked computers",
    "Cluster Computing": "using a group of computers as one system",
    "Grid Computing": "using distributed resources to solve large tasks",
    "Parallel Computing": "splitting tasks across multiple processors",
    "Concurrency": "executing multiple tasks simultaneously",
    "Virtual Reality": "computer generated simulation of real environments",
    "Augmented Reality": "overlaying digital content on the real world",
    "Mixed Reality": "combining real and virtual worlds",
    "Computer Graphics": "creating images with computers",
    "Rendering": "generating images from models",
    "Rasterization": "converting vector graphics to pixels",
    "Ray Tracing": "a rendering technique simulating light paths",
    "Shading": "techniques to represent lighting and texture",
    "GPU": "a processor specialized for graphics and parallel computing",
    "Shader": "a program controlling the rendering process",
    "Frame Buffer": "memory holding image data for display",
    "Pixel": "the smallest unit of a digital image",
    "Vector Graphics": "images based on mathematical formulas",
    "Bitmap": "an image made of pixels",
    "Resolution": "the number of pixels in an image",
    "Aspect Ratio": "the ratio of width to height of an image",
    "Compression": "reducing file size",
    "Lossless Compression": "compression without data loss",
    "Lossy Compression": "compression with some loss of quality",
    "JPEG": "a lossy image format",
    "PNG": "a lossless image format",
    "GIF": "an image format supporting animation",
    "MPEG": "a video compression standard",
    "MP3": "a compressed audio format",
    "WAV": "an uncompressed audio format",
    "Codec": "a device or software for encoding or decoding data",
    "Streaming": "transmitting data continuously for playback",
    "Buffering": "preloading data before playback",
    "Artificial Intelligence": "simulating human intelligence in machines",
    "Expert System": "AI based on a knowledge base and rules",
    "Knowledge Representation": "ways to represent facts and rules for AI",
    "Inference Engine": "software applying logic to knowledge base",
    "Search Algorithm": "an algorithm to explore problem spaces",
    "Heuristic": "a rule of thumb for problem solving",
    "Genetic Algorithm": "an optimization algorithm inspired by evolution",
    "Swarm Intelligence": "AI inspired by collective behavior of animals",
    "Fuzzy Logic": "reasoning with degrees of truth",
    "Natural Language Processing": "AI dealing with human language",
    "Tokenization": "splitting text into words or phrases",
    "Stemming": "reducing words to their base form",
    "Lemmatization": "reducing words to dictionary form",
    "Part of Speech Tagging": "assigning word categories like noun or verb",
    "Named Entity Recognition": "identifying entities like names and dates",
    "Sentiment Analysis": "determining sentiment in text",
    "Topic Modeling": "discovering themes in documents",
    "TF IDF": "a method to measure word importance in documents",
    "Bag of Words": "a model representing word counts",
    "Language Model": "a model predicting word sequences",
    "Perplexity": "a measure of how well a language model predicts",
    "BLEU Score": "a metric for machine translation quality",
    "ROUGE Score": "a metric for text summarization quality",
    "Word Embedding": "representing words as vectors",
    "Sentence Embedding": "representing sentences as vectors",
    "Transformer Model": "a neural network handling sequences in parallel",
    "Attention": "a mechanism focusing on important parts of input",
    "Self Attention": "attention mechanism relating words in a sentence",
    "Cross Attention": "attention between different sequences",
    "Encoder": "part of a transformer reading input",
    "Decoder": "part of a transformer generating output",
    "Pretraining": "training a model on large data before fine tuning",
    "Fine Tuning": "adapting a pretrained model to specific tasks",
    "Zero Shot Learning": "performing tasks without training examples",
    "Few Shot Learning": "performing tasks with few training examples",
    "Prompt Engineering": "designing inputs to guide model outputs",
    "Reinforcement Learning from Human Feedback": "training with AI and human guidance",
    "Speech Recognition": "converting speech into text",
    "Text to Speech": "converting text into speech",
    "Machine Translation": "automatically translating languages",
    "Summarization": "condensing long text into shorter form",
    "Information Retrieval": "finding relevant documents from a collection",
    "Search Engine": "software for searching large datasets",
    "Crawler": "a program that collects web pages",
    "Indexer": "a program that organizes web content",
    "Ranking": "ordering search results by relevance",
    "PageRank": "an algorithm ranking web pages by importance",
    "Click Through Rate": "percentage of users clicking a link",
    "Recommendation System": "a system suggesting items to users",
    "Collaborative Filtering": "recommending items based on user behavior",
    "Content Based Filtering": "recommending items similar to ones liked",
    "Hybrid Recommendation": "combining multiple recommendation methods",
    "Personalization": "adapting content to individual users",
    "User Profile": "data representing a user’s preferences",
    "Data Analytics": "analyzing raw data to find insights",
    "Big Data": "datasets too large for traditional processing",
    "Hadoop": "a framework for distributed storage and processing",
    "MapReduce": "a model for processing large datasets",
    "Spark": "a fast data processing engine",
    "Data Lake": "a storage system for raw data",
    "ETL": "extract, transform, load data process",
    "OLAP": "online analytical processing",
    "OLTP": "online transaction processing",
    "Business Intelligence": "analyzing business data for decisions",
    "Dashboard": "a visual display of data",
    "Data Visualization": "representing data graphically",
    "Bar Chart": "a graph with rectangular bars",
    "Line Chart": "a graph showing trends over time",
    "Pie Chart": "a circular chart divided into sectors",
    "Histogram": "a chart showing frequency distribution",
    "Scatter Plot": "a chart showing data points",
    "Heatmap": "a chart using color to show values",
    "Cybersecurity": "protecting systems from digital attacks",
    "Vulnerability": "a weakness in a system",
    "Exploit": "a method to take advantage of a vulnerability",
    "Malware": "malicious software",
    "Virus": "malware that replicates itself",
    "Worm": "malware spreading without human action",
    "Trojan": "malware disguised as legitimate software",
    "Ransomware": "malware encrypting files for ransom",
    "Spyware": "malware collecting information secretly",
    "Phishing": "fraudulent attempts to get sensitive information",
    "Denial of Service": "an attack overwhelming a system with traffic",
    "DDoS": "a denial of service attack from many sources",
    "Man in the Middle": "an attacker intercepting communication",
    "SQL Injection": "an attack inserting malicious SQL commands",
    "Cross Site Scripting": "an attack injecting scripts into web pages",
    "Buffer Overflow": "an attack exploiting memory overflow",
    "Brute Force Attack": "trying many passwords until correct",
    "Encryption": "converting data into unreadable form",
    "Decryption": "converting encrypted data back to original",
    "Symmetric Encryption": "encryption using one key for both encryption and decryption",
    "Asymmetric Encryption": "encryption using public and private keys",
    "Hash Function": "a function mapping data to fixed-size output",
    "Digital Signature": "a way to verify authenticity of messages",
    "Public Key Infrastructure": "a system managing digital certificates",
    "SSL": "a protocol for secure communication",
    "TLS": "an updated version of SSL",
    "Blockchain": "a distributed ledger of transactions",
    "Cryptocurrency": "a digital currency using cryptography",
    "Bitcoin": "the first decentralized cryptocurrency",
    "Ethereum": "a blockchain supporting smart contracts",
    "Smart Contract": "a program that runs on blockchain",
    "Consensus Algorithm": "a method to agree on blockchain state",
    "Proof of Work": "a consensus method requiring computation",
    "Proof of Stake": "a consensus method requiring ownership of coins",
    "Mining": "the process of adding transactions to blockchain",
    "Node": "a computer participating in blockchain",
    "Ledger": "a record of transactions",
    "Token": "a digital asset on blockchain",
    "Non Fungible Token": "a unique digital asset",
    "Smart Grid": "an intelligent electricity network",
    "Internet of Things": "devices connected to the internet",
    "Sensor": "a device that detects physical properties",
    "Actuator": "a device that performs actions",
    "Embedded Software": "software running on embedded systems",
    "Wearable Technology": "devices worn on the body with computing",
    "Smart Home": "a house with connected devices",
    "Autonomous Vehicle": "a self driving car",
    "Robotics": "engineering dealing with robots",
    "Drone": "an unmanned aerial vehicle",
    "3D Printing": "making objects by adding layers of material",
    "Nanotechnology": "technology working at atomic scale",
    "Quantum Computing": "computing using quantum mechanics",
    "Qubit": "the basic unit of quantum information",
    "Superposition": "a qubit being in multiple states",
    "Entanglement": "a quantum property linking qubits",
    "Quantum Gate": "an operation on qubits",
    "Quantum Algorithm": "an algorithm using quantum properties",
    "Shor's Algorithm": "a quantum algorithm for factoring numbers",
    "Grover's Algorithm": "a quantum algorithm for searching",
    "Quantum Supremacy": "quantum computers outperforming classical",
    "Cloud Native": "applications built to run in the cloud",
    "Microservices": "architectural style dividing applications into small services",
    "API": "a way for programs to communicate",
    "REST": "an API style using HTTP",
    "GraphQL": "an API query language",
    "SOAP": "a protocol for exchanging structured information",
    "WebSocket": "a protocol for real time communication",
    "gRPC": "a high performance RPC framework",
    "Middleware": "software between OS and applications",
    "Service Oriented Architecture": "an architecture organizing software as services",
    "Event Driven Architecture": "an architecture reacting to events",
    "Message Queue": "a system for exchanging messages between processes",
    "Kafka": "a distributed event streaming platform",
    "RabbitMQ": "a message broker software",
    "ActiveMQ": "a message oriented middleware",
    "CI/CD": "continuous integration and deployment",
    "DevOps": "a culture combining development and operations",
    "Agile": "a software development method with iterations",
    "Scrum": "an agile framework with sprints",
    "Kanban": "a method to visualize and manage work",
    "Lean Software Development": "a method focusing on efficiency",
    "Extreme Programming": "an agile method emphasizing coding practices",

    "Software Development Life Cycle": "a process for planning, creating, testing, and deploying software",
    "Requirement Analysis": "gathering and defining software requirements",
    "System Design": "planning the architecture of a system",
    "Implementation": "the phase of writing and coding the program",
    "Testing": "the process of finding and fixing software bugs",
    "Unit Testing": "testing individual components of software",
    "Integration Testing": "testing combined modules of software",
    "System Testing": "testing the complete integrated system",
    "Acceptance Testing": "testing to verify the system meets requirements",
    "Black Box Testing": "testing without knowledge of internal code",
    "White Box Testing": "testing with knowledge of internal code",
    "Gray Box Testing": "testing with partial knowledge of internal code",
    "Regression Testing": "testing to check new changes do not break old features",
    "Smoke Testing": "basic testing to ensure critical functions work",
    "Sanity Testing": "testing specific functions after changes",
    "Performance Testing": "testing system speed and responsiveness",
    "Load Testing": "testing system behavior under expected load",
    "Stress Testing": "testing system behavior under extreme load",
    "Scalability Testing": "testing if system can scale up with demand",
    "Security Testing": "testing to ensure system is secure",
    "Usability Testing": "testing ease of use of software",
    "Alpha Testing": "internal testing before release",
    "Beta Testing": "testing by real users before final release",
    "Deployment": "delivering software to users",
    "Maintenance": "updating and fixing software after release",
    "Version Control": "a system to manage changes in code",
    "Git": "a distributed version control system",
    "GitHub": "a platform for hosting Git repositories",
    "Branch": "a parallel version of a repository",
    "Merge": "combining changes from different branches",
    "Commit": "a saved change in a repository",
    "Pull Request": "a request to merge changes into main branch",
    "Fork": "a copy of a repository for independent work",
    "CI": "continuous integration, automatic testing and building",
    "CD": "continuous delivery/deployment, automatic release process",
    "Build Automation": "automatically compiling and packaging code",
    "Container": "a lightweight package with application and dependencies",
    "Orchestration": "managing containers at scale",
    "Monitoring": "tracking system performance and errors",
    "Logging": "recording system events and messages",
    "Alerting": "notifying about issues in systems",
    "Scripting Language": "a language used to automate tasks",
    "Compiled Language": "a language that is converted to machine code before execution",
    "Interpreted Language": "a language executed line by line",
    "Statically Typed Language": "a language where variable types are declared at compile time",
    "Dynamically Typed Language": "a language where variable types are determined at runtime",
    "Strongly Typed Language": "a language that enforces strict type rules",
    "Weakly Typed Language": "a language that allows flexible type conversions",
    "Functional Programming": "a paradigm treating computation as evaluation of functions",
    "Declarative Programming": "a style where you specify what to do, not how",
    "Imperative Programming": "a style focusing on how to perform tasks",
    "Logic Programming": "a paradigm based on formal logic",
    "Event Driven Programming": "programming where flow is determined by events",
    "Concurrency Model": "the way a language handles concurrent tasks",
    "Garbage Collection": "automatic memory management",
    "Memory Leak": "unused memory that is not released",
    "Pointer": "a variable that stores a memory address",
    "Reference": "an alias for another variable",
    "Null Pointer": "a pointer that does not point to valid memory",
    "Segmentation Fault": "an error when accessing invalid memory",
    "Buffer": "a temporary storage area",
    "Stack Overflow": "an error caused by exceeding stack size",
    "Heap Overflow": "an error caused by exceeding heap size",
    "Bit Manipulation": "directly working with individual bits",
    "Endianness": "the order of bytes in memory",
    "Concurrency Bug": "an error caused by incorrect handling of concurrency",
    "Race Condition": "a problem when processes depend on timing",
    "Atomic Operation": "an operation that completes without interruption",
    "Lock": "a mechanism to control access to resources",
    "Deadlock Detection": "finding processes stuck in deadlock",
    "Starvation Prevention": "techniques to ensure all processes get resources",
    "Scheduling Algorithm": "rules for deciding process execution order",
    "Round Robin": "a scheduling method giving equal time to each process",
    "Priority Scheduling": "executing processes based on priority",
    "Multilevel Queue": "scheduling with multiple queues of priorities",
    "Multilevel Feedback Queue": "a dynamic scheduling with process movement between queues",
    "File Allocation Table": "a system for organizing files on disk",
    "Inode": "a data structure storing file metadata",
    "Journaling File System": "a file system that logs changes",
    "NTFS": "Windows file system supporting journaling",
    "EXT4": "Linux file system supporting journaling",
    "HDFS": "Hadoop distributed file system",
    "RAID": "a method for storing data across multiple disks",
    "RAID 0": "disk striping without redundancy",
    "RAID 1": "disk mirroring with redundancy",
    "RAID 5": "striping with distributed parity",
    "RAID 10": "a combination of mirroring and striping",
    "Cloud Security": "protecting data and apps in the cloud",
    "Identity Management": "ensuring correct users access systems",
    "Access Control": "restricting access to resources",
    "Authentication": "verifying user identity",
    "Authorization": "granting permissions to users",
    "Single Sign On": "one login for multiple systems",
    "Multi Factor Authentication": "using multiple proofs of identity",
    "OAuth": "a protocol for secure authorization",
    "OpenID Connect": "an authentication layer on OAuth",
    "SAML": "a standard for exchanging authentication data",
    "Biometric Authentication": "using physical traits for authentication",
    "CAPTCHA": "a test to distinguish humans from bots",
    "Data Privacy": "protecting personal information",
    "GDPR": "European regulation for data protection",
    "HIPAA": "US regulation for health data privacy",
    "PCI DSS": "security standard for payment data",
    "Penetration Testing": "simulating attacks to find vulnerabilities",
    "Red Team": "offensive security professionals",
    "Blue Team": "defensive security professionals",
    "Purple Team": "collaboration of red and blue teams",
    "Forensics": "investigating digital crimes",
    "Incident Response": "handling security breaches",
    "Disaster Recovery": "restoring systems after failure",
    "Business Continuity": "maintaining operations during disruptions",
    "Data Backup": "copying data to prevent loss",
    "Cold Backup": "backup system kept offline",
    "Hot Backup": "backup system kept online",
    "Incremental Backup": "backing up only changed data",
    "Differential Backup": "backing up data since last full backup",
    "Cloud Backup": "storing backup in the cloud",
    "Mobile Computing": "using computers on mobile devices",
    "Smartphone": "a handheld device with computing capabilities",
    "Tablet": "a portable touchscreen device",
    "Wearable Device": "electronics worn on the body",
    "Mobile App": "software designed for mobile devices",
    "Native App": "app built for a specific platform",
    "Hybrid App": "app combining web and native elements",
    "Cross Platform App": "app running on multiple platforms",
    "Responsive Design": "design adapting to screen sizes",
    "Progressive Web App": "web apps with native-like features",
    "App Store": "a marketplace for apps",
    "Google Play": "Android app store",
    "Apple App Store": "iOS app store",
    "Mobile Security": "protecting mobile devices and data",
    "SIM Card": "a chip storing mobile subscriber identity",
    "IMEI": "a unique number identifying a mobile device",
    "5G": "the fifth generation of mobile networks",
    "4G": "the fourth generation of mobile networks",
    "3G": "the third generation of mobile networks",
    "SMS": "short message service",
    "MMS": "multimedia messaging service",
    "VoIP": "voice communication over internet",
    "Streaming Media": "continuous delivery of audio and video",
    "Digital Twin": "a digital copy of a physical object",
    "Edge AI": "AI running on edge devices",
    "Federated Learning": "training AI across devices without centralizing data",
    "Explainable AI": "AI methods providing understandable decisions",
    "AI Ethics": "principles for responsible AI use",
    "Bias in AI": "systematic errors caused by unfair training data",
    "Fairness in AI": "ensuring equitable AI outcomes",
    "Autonomous Systems": "systems operating without human control",
    "Swarm Robotics": "robots working collectively",
    "Human Computer Interaction": "study of interaction between humans and computers",
    "User Interface": "the part of software users interact with",
    "Graphical User Interface": "a visual way to interact with computers",
    "Command Line Interface": "a text-based way to interact with computers",
    "Voice User Interface": "an interface using spoken language",
    "Haptic Feedback": "using touch to communicate with users",
    "Accessibility": "making systems usable by everyone",
    "Responsive UI": "interfaces that adapt to screen size",
    "UI Design": "the design of user interfaces",
    "UX Design": "the design of user experience",
    "Wireframe": "a basic layout of a user interface",
    "Prototype": "a model of software for testing ideas",
    "A/B Testing": "comparing two versions of a system",
    "User Testing": "observing real users interacting with a system",
    "Feedback Loop": "using results to improve systems",
    "Gamification": "using game elements in non-game contexts",
    "Ergonomics": "designing for human comfort and efficiency",
    "Information Architecture": "organizing information for usability",
    "Interaction Design": "designing how users interact with systems",
    "Cognitive Load": "mental effort required to use a system",
    "Heuristic Evaluation": "experts reviewing usability",
    "Eye Tracking": "measuring where users look on screens",
    "Heatmap Analysis": "visualizing areas of user interaction",
    "Usability Heuristics": "general rules for designing usable systems",
    "Dark Patterns": "tricks in UI to manipulate users",
    "Open Source Software": "software with publicly available code",
    "Closed Source Software": "software with proprietary code",
    "Freeware": "software available at no cost",
    "Shareware": "software free for limited use",
    "Proprietary Software": "software owned by an individual or company",
    "Licensing": "permissions to use software",
    "GPL": "general public license for free software",
    "MIT License": "a permissive free software license",
    "Apache License": "a permissive open-source license",
    "Creative Commons": "licenses for creative works",
    "Software Patent": "legal protection for software inventions",
    "Intellectual Property": "legal rights for creations of the mind",
    "Piracy": "illegal copying or distribution of software",
    "Reverse Engineering": "analyzing software to recreate it",
    "Decompilation": "converting binary code back to source-like form",
    "Obfuscation": "making code harder to understand",
    "Code Review": "examining code for errors and improvements",
    "Pair Programming": "two developers writing code together",
    "Refactoring": "improving code without changing functionality",
    "Technical Debt": "extra work caused by poor design",
    "Design Pattern": "a reusable solution to common problems",
    "Singleton Pattern": "ensures only one instance of a class",
    "Factory Pattern": "creates objects without specifying exact classes",
    "Observer Pattern": "defines one-to-many dependencies",
    "Decorator Pattern": "adds behavior to objects dynamically",
    "Adapter Pattern": "allows incompatible interfaces to work together",
    "Facade Pattern": "provides a simplified interface",
    "Proxy Pattern": "controls access to objects",
    "Strategy Pattern": "defines a family of algorithms to interchange",
    "Template Pattern": "defines the skeleton of an algorithm",
    "MVC Pattern": "divides app into model, view, and controller",
    "MVVM Pattern": "divides app into model, view, and view-model",
    "Clean Architecture": "an approach to keep systems maintainable",
    "Domain Driven Design": "building systems based on domain logic",
    "Service Layer": "an architectural layer handling business logic",
    "Repository Pattern": "abstracts data access from business logic",
    "Event Sourcing": "storing system state as sequence of events",
    "Command Query Responsibility Segregation": "separating read and write operations",
    "Hexagonal Architecture": "an architecture promoting flexibility",
    "Twelve Factor App": "guidelines for building SaaS apps",


}


In [5]:
def simplify_jargon(text):
    simplified_text = text
    for jargon, meaning in glossary.items():
        simplified_text = re.sub(rf"\b{jargon}\b", f"{jargon} ({meaning})", simplified_text, flags=re.IGNORECASE)
    return simplified_text


In [6]:
paraphraser = pipeline("text2text-generation", model="t5-base")

def paraphrase_text(text):
    simplified = paraphraser(f"simplify: {text}", max_length=200, do_sample=False)[0]['generated_text']
    return simplified


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Device set to use cpu


In [7]:
def engineering_simplifier(text, use_paraphrasing=False):
    # Step 1: Replace jargon
    rule_based = simplify_jargon(text)

    # Step 2: Paraphrase if selected
    if use_paraphrasing:
        return paraphrase_text(rule_based)
    else:
        return rule_based


In [8]:
with gr.Blocks() as demo:
    gr.Markdown("## 🛠️ Engineering Jargon Simplifier")

    inp = gr.Textbox(lines=5, placeholder="Paste your engineering text here...")
    chk = gr.Checkbox(label="Use Paraphrasing (AI)")
    out = gr.Textbox(lines=5, label="Simplified Text")

    btn = gr.Button("Simplify")
    btn.click(fn=engineering_simplifier, inputs=[inp, chk], outputs=out)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d80aebaacf50bd1210.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


📝 Example Sentences Using Batch 1 (1–200)

The operating system relies on process scheduling algorithms like Round Robin and Priority Scheduling to manage concurrency, ensuring that no deadlock or starvation occurs even when multiple programs compete for CPU resources.

In machine learning, gradient descent is often combined with regularization techniques to prevent overfitting, while embeddings and transformer models make natural language processing tasks like translation and sentiment analysis more effective.

Distributed databases such as NoSQL systems apply sharding and replication to improve scalability and fault tolerance, while maintaining ACID properties through sophisticated transaction mechanisms.

📝 Example Sentences Using Batch 2 (201–400)

Cloud computing platforms like AWS and Azure use virtualization and containerization technologies, along with orchestration tools such as Kubernetes, to provide scalable microservices that are monitored with logging and alerting systems.

In big data pipelines, raw input is processed with ETL tools, stored in distributed file systems like HDFS, and analyzed using frameworks such as Spark, while visualization dashboards like Tableau help decision-makers interpret the results.

Blockchain applications leverage smart contracts and consensus mechanisms like Proof of Stake or Proof of Work to ensure security and transparency in decentralized environments, while cryptographic hash functions guarantee immutability of the data.

📝 Example Sentences Using Batch 3 (401–600)

Software engineers often follow the Software Development Life Cycle, beginning with requirement analysis and system design, followed by implementation, testing phases such as unit and regression testing, and finally deployment and maintenance with version control tools like Git.

Security teams use penetration testing, multi-factor authentication, and incident response strategies to protect cloud-based infrastructures, while regulatory frameworks like GDPR and HIPAA ensure data privacy and compliance with legal standards.

Modern applications adopt clean architecture and design patterns such as Singleton, Factory, and Observer, combined with CI/CD pipelines, containerization, and monitoring systems, to deliver scalable, maintainable, and resilient enterprise solutions.

In human-computer interaction, usability heuristics, cognitive load analysis, and eye-tracking heatmap studies are performed to improve the graphical user interface, while accessibility principles ensure that applications remain inclusive for all types of users.